<a href="https://colab.research.google.com/github/SaZikang/data-science-toolkit/blob/main/datasciencetutorial3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial 03: Tidy Data & Data Characteristics
**Course:** Data Science / Analytics  
**Topic:** Tidy Data Principles, Dataset Characteristics, Missing Value Diagnostics, Feature Identification, and Analytical Inquiry  
**Author:** Sa Zikang  
**Repository:** [SaZikang/data-science-toolkit](https://github.com/SaZikang/data-science-toolkit)

---
### Tutorial Objectives:
This notebook comprehensively addresses all 9 tasks and questions for **Tutorial 03**:
1. Load a real-world dataset on Colab/Kaggle and visualize key distributions.
2. Evaluate data quality and sample size sufficiency.
3. Quantify features and assess feature adequacy.
4. Quantify samples/observations and assess observational sufficiency.
5. Identify and classify dataset datatypes.
6. Diagnose null values: column-wise counts, percentages, and exact row locations.
7. Disentangle input features ($X$) from the target output ($y$).
8. Deep-dive into missing values, missingness mechanisms (MCAR, MAR, MNAR), and row positions.
9. Formulate domain questions and assess dataset suitability.


In [ ]:
# 0. Setup and Library Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Suppress minor warnings for cleaner outputs
warnings.filterwarnings('ignore')

# Configure visualization aesthetics
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("✓ Core libraries loaded successfully: pandas, numpy, matplotlib, seaborn.")


---
## Question 1: Find some real dataset. Load it on Colab, Kaggle etc. Try to visualize some of it

### Dataset Selection: RMS Titanic Passenger Manifest
For this tutorial, we analyze the classic, real-world **Titanic Dataset**, documenting the demographic, socioeconomic, and survival status of 891 passengers aboard the ill-fated maiden voyage of the RMS Titanic in April 1912.

- **Source:** Seaborn official repository / Kaggle Titanic: Machine Learning from Disaster
- **Data URL:** `https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv`
- **Tidy Data Structure:** 
  1. Each variable forms a column.
  2. Each observation forms a row (one individual passenger).
  3. Each type of observational unit forms a table.


In [ ]:
# 1.1 Load the Real-World Titanic Dataset
dataset_url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
df = pd.read_csv(dataset_url)

print(f"Dataset successfully loaded!")
print(f"Dimensions: {df.shape[0]} rows (passengers) x {df.shape[1]} columns (attributes)")
print("\nFirst 5 rows of the dataset:")
df.head()


In [ ]:
# 1.2 Data Visualization: Key Distributions and Relationships
fig, axes = plt.subplots(2, 2, figsize=(15, 11))

# Subplot 1: Overall Survival Distribution
sns.countplot(
    data=df, 
    x='survived', 
    ax=axes[0, 0], 
    hue='survived', 
    palette=['#e74c3c', '#2ecc71'], 
    legend=False
)
axes[0, 0].set_title('Figure 1A: Overall Survival Distribution', fontsize=13, fontweight='bold')
axes[0, 0].set_xticklabels(['Perished (0)', 'Survived (1)'])
axes[0, 0].set_xlabel('Survival Outcome')
axes[0, 0].set_ylabel('Passenger Count')
for p in axes[0, 0].patches:
    axes[0, 0].annotate(f'{int(p.get_height())}', 
                        (p.get_x() + p.get_width() / 2., p.get_height()),
                        ha='center', va='center', xytext=(0, 6), textcoords='offset points', fontweight='bold')

# Subplot 2: Survival Rate by Gender
sns.barplot(
    data=df, 
    x='sex', 
    y='survived', 
    ax=axes[0, 1], 
    hue='sex', 
    palette=['#3498db', '#e84393'], 
    legend=False
)
axes[0, 1].set_title('Figure 1B: Survival Rate by Gender ("Women and Children First")', fontsize=13, fontweight='bold')
axes[0, 1].set_xlabel('Sex')
axes[0, 1].set_ylabel('Proportion Survived')
axes[0, 1].set_ylim(0, 1.0)
for p in axes[0, 1].patches:
    axes[0, 1].annotate(f'{p.get_height()*100:.1f}%', 
                        (p.get_x() + p.get_width() / 2., p.get_height()),
                        ha='center', va='center', xytext=(0, 6), textcoords='offset points', fontweight='bold')

# Subplot 3: Survival Rate by Socioeconomic Ticket Class
sns.barplot(
    data=df, 
    x='class', 
    y='survived', 
    order=['First', 'Second', 'Third'], 
    ax=axes[1, 0], 
    palette='Blues_d'
)
axes[1, 0].set_title('Figure 1C: Survival Rate by Passenger Class', fontsize=13, fontweight='bold')
axes[1, 0].set_xlabel('Passenger Class')
axes[1, 0].set_ylabel('Proportion Survived')
axes[1, 0].set_ylim(0, 1.0)
for p in axes[1, 0].patches:
    axes[1, 0].annotate(f'{p.get_height()*100:.1f}%', 
                        (p.get_x() + p.get_width() / 2., p.get_height()),
                        ha='center', va='center', xytext=(0, 6), textcoords='offset points', fontweight='bold')

# Subplot 4: Age Distribution Stratified by Survival Outcome
sns.histplot(
    data=df, 
    x='age', 
    hue='survived', 
    multiple='stack', 
    kde=True, 
    ax=axes[1, 1], 
    palette=['#e74c3c', '#2ecc71']
)
axes[1, 1].set_title('Figure 1D: Age Distribution by Survival Outcome', fontsize=13, fontweight='bold')
axes[1, 1].set_xlabel('Age (Years)')
axes[1, 1].set_ylabel('Passenger Count')
axes[1, 1].legend(title='Outcome', labels=['Survived', 'Perished'])

plt.tight_layout()
plt.show()


In [ ]:
# 1.3 Additional Visualization: Fare vs Passenger Class and Survival
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x='class', y='fare', hue='survived', palette=['#e74c3c', '#2ecc71'], showfliers=False)
plt.title('Figure 1E: Ticket Fare Distribution by Class and Survival (Outliers Hidden)', fontsize=13, fontweight='bold')
plt.xlabel('Ticket Class')
plt.ylabel('Fare Paid (£ British Pounds)')
plt.legend(title='Status', labels=['Perished', 'Survived'])
plt.show()


---
## Question 2: Evaluate if the data is good or not. Does it have enough samples?

### Answer:
#### 1. Evaluation of Data Quality ("Is the data good?"):
The dataset exhibits **high overall structural quality** with specific domain limitations:
* **Strengths:**
  - **Tidy Format:** Conforms strictly to Hadley Wickham’s Tidy Data principles: each row is an individual observation (passenger), and each column is a unique measured variable.
  - **Historical Authenticity:** The observations reflect real historical human outcomes rather than synthetic data.
  - **Clear Target Variable:** Binary ground truth (`survived`: 0 = Perished, 1 = Survived) with no label noise or ambiguity.
  - **Diversity of Predictors:** Contains demographic (`sex`, `age`), socioeconomic (`pclass`, `fare`), and relational/social (`sibsp`, `parch`, `alone`) variables.
* **Limitations / Data Quality Flaws:**
  - **High Missingness in Cabin/Deck:** The `deck` variable is 77.2% missing ($688 / 891$), rendering it unusable without imputation or transformation.
  - **Information Redundancy:** Multiple columns represent identical underlying information (`pclass` $\leftrightarrow$ `class`; `survived` $\leftrightarrow$ `alive`; `embarked` $\leftrightarrow$ `embark_town`).
  - **Historical Recording Bias:** Age records for 177 passengers (19.9%) were missing, predominantly in 3rd class, reflecting historical record-keeping disparity.

#### 2. Sample Size Evaluation ("Does it have enough samples?"):
* **For Exploratory Data Analysis & Classical Machine Learning:** **YES, 891 samples is sufficient.**  
  With 891 observations and an events-per-variable ratio $> 20:1$ for binary classification (342 survivors vs 549 non-survivors across ~10 independent predictors), standard algorithms (Logistic Regression, Decision Trees, Random Forests, Naive Bayes, SVMs) train reliably with stable cross-validation scores (~80–84% accuracy) without extreme risk of variance collapse.
* **For Deep Learning / Neural Networks:** **NO, 891 samples is relatively small.**  
  Modern deep neural networks require thousands to millions of samples to generalize without severe overfitting.


In [ ]:
# 2.1 Code Diagnostic for Question 2
print("=== DATA QUALITY METRICS ===")
print(f"Total Rows (Observations): {len(df)}")
print(f"Total Columns: {df.shape[1]}")
print(f"Exact Duplicate Rows: {df.duplicated().sum()}")
print(f"Completely Clean Rows (Zero Missing Values): {df.dropna().shape[0]} ({(df.dropna().shape[0]/len(df))*100:.1f}%)")
print(f"Total Missing Cells across all data: {df.isnull().sum().sum()} / {df.size} ({df.isnull().sum().sum()/df.size*100:.2f}%)")


---
## Question 3: How many features does it have? Is it enough?

### Answer:
#### 1. Number of Features:
* The dataset has **15 total columns**.
* Since `survived` (or its duplicate `alive`) is the target output variable, there are **14 feature columns** provided in the raw table:
  1. `pclass` (Ticket class: 1, 2, 3)
  2. `sex` (Biological sex: male, female)
  3. `age` (Passenger age in years)
  4. `sibsp` (Number of siblings / spouses aboard)
  5. `parch` (Number of parents / children aboard)
  6. `fare` (Passenger ticket fare in British Pounds)
  7. `embarked` (Port of Embarkation code: C, Q, S)
  8. `class` (Ticket class label: First, Second, Third) - *redundant with `pclass`*
  9. `who` (Demographic persona: man, woman, child)
  10. `adult_male` (Boolean flag: True if adult male)
  11. `deck` (Cabin deck letter: A, B, C, D, E, F, G)
  12. `embark_town` (Port name: Cherbourg, Queenstown, Southampton) - *redundant with `embarked`*
  13. `alive` (Outcome text: yes, no) - *redundant with `survived`*
  14. `alone` (Boolean flag: True if traveling without family)
* After removing the duplicate / alias columns, there are **9 core non-redundant predictive features**:  
  `pclass`, `sex`, `age`, `sibsp`, `parch`, `fare`, `embarked`, `who`, `alone`.

#### 2. Is it enough?
* **YES, for the primary objective of predicting maritime survival and testing historical sociological hypotheses.**  
  These 9 core features span the four critical causal dimensions of the event:
  1. *Social protocol & evacuation priority:* `sex`, `age`, `who`
  2. *Socioeconomic status & physical accommodation placement:* `pclass`, `fare`
  3. *Family coordination & encumbrance:* `sibsp`, `parch`, `alone`
  4. *Logistical boarding point:* `embarked`
* **Additional features that would be ideal (if available):**
  - Cabin distance to the boat deck / evacuation stairwells.
  - Lifeboat assignment number and launch timestamp.
  - Ability to swim and physical mobility.


In [ ]:
# 3.1 Code Diagnostic for Question 3
print("=== FEATURE ENUMERATION ===")
feature_cols = [c for c in df.columns if c not in ['survived', 'alive']]
print(f"Total raw columns: {df.shape[1]}")
print(f"Target column(s): ['survived', 'alive']")
print(f"Input feature columns: {len(feature_cols)}")
print("\nFeature Breakdown:")
for i, col in enumerate(feature_cols, 1):
    redundant = " (Redundant alias)" if col in ['class', 'embark_town'] else ""
    print(f"  {i:2d}. {col:12s} | Type: {str(df[col].dtype):8s}{redundant}")


---
## Question 4: How many samples/observation does it have? Is it enough?

### Answer:
#### 1. Number of Samples / Observations:
* The dataset contains **891 individual samples (rows)**.
* Each sample corresponds to one verified passenger aboard the Titanic.

#### 2. Is it enough?
* **YES, 891 samples is statistically sufficient for this analysis:**
  1. **Statistical Power:** A sample size of $N = 891$ provides adequate statistical power ($> 0.99$ at $\alpha = 0.05$) to detect medium and small effect sizes in survival probability differences across gender, class, and age groups.
  2. **Class Balance:** The target class distribution is reasonably balanced:
     - Non-survivors ($0$): **549 passengers (61.6%)**
     - Survivors ($1$): **342 passengers (38.4%)**  
     Because the minority class has $342$ occurrences, it does not suffer from extreme class imbalance (such as fraud detection with $0.1\%$ positive cases).
  3. **Events Per Variable (EPV):** In logistic regression, standard statistical practice requires an EPV ratio of at least $10:1$ to $15:1$. With $342$ survival events and $9$ core features, the EPV ratio is $342 / 9 \approx 38:1$, well above the accepted threshold to prevent severe overfitting.


In [ ]:
# 4.1 Code Diagnostic for Question 4
n_samples = len(df)
surv_counts = df['survived'].value_counts()
surv_pcts = df['survived'].value_counts(normalize=True) * 100

print(f"Total Observations / Samples: {n_samples}")
print(f"  - Perished (0): {surv_counts[0]} ({surv_pcts[0]:.2f}%)")
print(f"  - Survived (1): {surv_counts[1]} ({surv_pcts[1]:.2f}%)")
print(f"\nObservation-to-Feature Ratio: {n_samples / len(feature_cols):.1f} observations per feature")


---
## Question 5: What are the datatypes does the dataset consist of?

### Answer:
The dataset consists of **4 distinct data types**:
1. **`int64` (Integer / Discrete Numeric):** 
   - `survived` (0 or 1 binary indicator)
   - `pclass` (1, 2, or 3 representing ticket tier)
   - `sibsp` (count of siblings/spouses: 0 to 8)
   - `parch` (count of parents/children: 0 to 6)
2. **`float64` (Floating-Point / Continuous Numeric):**
   - `age` (passenger age in fractional years, e.g., 0.42 to 80.0)
   - `fare` (monetary ticket cost in British pounds, e.g., £0.00 to £512.33)
3. **`object` (String / Categorical):**
   - `sex` ('male', 'female') - Nominal Categorical
   - `embarked` ('S', 'C', 'Q') - Nominal Categorical
   - `class` ('First', 'Second', 'Third') - Ordinal Categorical
   - `who` ('man', 'woman', 'child') - Nominal Categorical
   - `deck` ('A', 'B', 'C', 'D', 'E', 'F', 'G') - Nominal/Ordinal Categorical
   - `embark_town` ('Southampton', 'Cherbourg', 'Queenstown') - Nominal Categorical
   - `alive` ('no', 'yes') - Binary Categorical
4. **`bool` (Boolean / Binary Logical):**
   - `adult_male` (True, False)
   - `alone` (True, False)


In [ ]:
# 5.1 Code Diagnostic for Question 5: Data Types Table
dtype_summary = pd.DataFrame({
    'Column Name': df.columns,
    'Pandas Dtype': df.dtypes.astype(str),
    'Data Nature': [
        'Binary Target (Discrete)', 'Ordinal Categorical (Encoded)', 'Nominal Categorical',
        'Continuous Numeric', 'Discrete Numeric (Count)', 'Discrete Numeric (Count)',
        'Continuous Numeric (Currency)', 'Nominal Categorical', 'Ordinal Categorical (Text)',
        'Nominal Categorical', 'Boolean Flag', 'Nominal/Ordinal Categorical',
        'Nominal Categorical', 'Binary Target (Text)', 'Boolean Flag'
    ],
    'Non-Null Count': df.notnull().sum(),
    'Null Count': df.isnull().sum(),
    'Unique Values': df.nunique(),
    'Example Value': df.iloc[1].values
}).reset_index(drop=True)

display(dtype_summary)

print("\nCounts of Columns by Dtype:")
print(df.dtypes.value_counts())


---
## Question 6: How many columns have null values? If yes, which column, which rows

### Answer:
* **Number of columns with null values:** Exactly **4 columns** have null values.
* **The 4 Columns and their null distributions:**
  1. **`deck`**: **688 null rows** ($77.22\%$ missing)
  2. **`age`**: **177 null rows** ($19.87\%$ missing)
  3. **`embarked`**: **2 null rows** ($0.22\%$ missing) $\rightarrow$ **Row 61 and Row 829**
  4. **`embark_town`**: **2 null rows** ($0.22\%$ missing) $\rightarrow$ **Row 61 and Row 829**

#### Exact Row Identifications:
* For `embarked` and `embark_town`: Exactly **Row Index 61** and **Row Index 829** (both were First Class passengers: Mrs. Stone and her maid Miss Icard, traveling together in cabin B28 on ticket 113572).
* For `age`: 177 rows, starting at indices: `[5, 17, 19, 26, 28, 29, 31, 32, 36, 42, 45, 46, 47, 48, 55, ...]`.
* For `deck`: 688 rows, starting at indices: `[0, 2, 4, 5, 7, 8, 9, 12, 13, 14, 15, 16, 17, 18, 19, 20, ...]`.


In [ ]:
# 6.1 Code Diagnostic for Question 6: Identifying Null Columns and Exact Rows
null_series = df.isnull().sum()
null_cols = null_series[null_series > 0]

print(f"Total columns with null values: {len(null_cols)}")
print("=" * 70)

for col in null_cols.index:
    null_row_indices = df[df[col].isnull()].index.tolist()
    pct = (len(null_row_indices) / len(df)) * 100
    print(f"Column '{col}':")
    print(f"  • Total Nulls: {len(null_row_indices)} / {len(df)} ({pct:.2f}%)")
    if len(null_row_indices) <= 10:
        print(f"  • EXACT ROW INDICES: {null_row_indices}")
    else:
        print(f"  • First 20 Row Indices: {null_row_indices[:20]}")
        print(f"  • Last 10 Row Indices:  {null_row_indices[-10:]}")
    print("-" * 70)


In [ ]:
# 6.2 Deep Inspection of Rows with Missing 'embarked'
print("Inspecting the 2 exact rows where 'embarked' and 'embark_town' are NULL:")
display(df.loc[[61, 829]])


---
## Question 7: Which column is the feature, which is the target output?

### Answer:
In supervised machine learning and statistical modeling:
* **Target Output ($y$):**
  - **`survived`**: The primary ground-truth dependent variable. It is a binary indicator where:
    - $1$ = Survived the disaster
    - $0$ = Perished in the disaster
  - *(Note: `alive` is an exact semantic duplicate of `survived` with string values `'yes'` and `'no'`)*.

* **Input Features ($X$):**
  All remaining columns that serve as independent predictor variables:
  | Category | Feature Columns | Description |
  | :--- | :--- | :--- |
  | **Demographic** | `sex`, `age`, `who`, `adult_male` | Biological sex, chronological age, persona category |
  | **Socioeconomic** | `pclass`, `class`, `fare` | Passenger ticket class tier, monetary fare paid |
  | **Family Structure** | `sibsp`, `parch`, `alone` | Sibling/spouse count, parent/child count, solo travel flag |
  | **Logistical / Spatial** | `embarked`, `embark_town`, `deck` | Port of boarding, cabin deck assignment |


In [ ]:
# 7.1 Code Diagnostic for Question 7: Partitioning X and y
target_col = 'survived'
alias_target = 'alive'
feature_cols = [c for c in df.columns if c not in [target_col, alias_target]]

y = df[target_col]
X = df[feature_cols]

print(f"TARGET OUTPUT (y): '{target_col}'")
print(f"Target Values: {y.unique().tolist()} -> Value distribution: {dict(y.value_counts())}")
print(f"\nFEATURE MATRIX (X): {X.shape[1]} features across {X.shape[0]} observations")
print(f"Features: {list(X.columns)}")


---
## Question 8: Does it have missing values? If yes, which column, which row.

### Answer:
#### 1. Yes, the dataset contains missing values.
Across the entire table of $891 \times 15 = 13,365$ cells, there are **869 total missing values** distributed across **4 columns**:

| Column Name | Missing Count | % Missing | Missingness Mechanism | Exact Row Indices |
| :--- | :---: | :---: | :--- | :--- |
| **`embarked`** | 2 | 0.22% | **MCAR** (Missing Completely at Random) | **Row 61, Row 829** |
| **`embark_town`** | 2 | 0.22% | **MCAR** (Missing Completely at Random) | **Row 61, Row 829** |
| **`age`** | 177 | 19.87% | **MAR** (Missing at Random) | 177 rows: `[5, 17, 19, 26, 28, 29, 31, 32, 36, 42, 45, ...]` |
| **`deck`** | 688 | 77.22% | **MNAR** (Missing Not at Random) | 688 rows: `[0, 2, 4, 5, 7, 8, 9, 12, 13, 14, 15, 16, ...]` |

#### 2. Diagnostic Analysis of Missingness Mechanisms:
1. **`embarked` / `embark_town` (MCAR):**
   Missing completely at random. Because only 2 passengers are affected ($0.22\%$), standard practice is either simple mode imputation (`'S'`) or dropping those 2 records.
2. **`age` (MAR):**
   Missing at random. Missingness correlates with ticket class and ticket fare (poorer 3rd-class passengers had less documented birth records than 1st-class passengers). Appropriate remediation: impute using median age grouped by `pclass` and `sex` or an iterative MICE imputer.
3. **`deck` (MNAR):**
   Missing not at random. Cabins with assigned deck letters were exclusively designated for First Class and selected Second Class passengers. Third-class passengers stayed in steerage dormitories without assigned deck letters. Thus, a missing deck value is informative (it indicates non-luxury status). Standard remediation: create an explicit category `'Unknown'` or `'Deck_Missing'`.


In [ ]:
# 8.1 Code Diagnostic for Question 8: Missingness Heatmap and Exact Statistics
plt.figure(figsize=(12, 4))
sns.heatmap(df.isnull(), cbar=True, cmap='cividis', yticklabels=False)
plt.title('Figure 8: Matrix Heatmap of Missing Values (Yellow = Missing, Navy = Present)', fontsize=13, fontweight='bold')
plt.show()

# Verification of all 4 columns
missing_stats = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing Percentage (%)': (df.isnull().sum() / len(df)) * 100,
    'Mechanism': [
        'Complete', 'Complete', 'Complete', 'MAR (Class-correlated)', 'Complete', 'Complete',
        'Complete', 'MCAR (Random omissions)', 'Complete', 'Complete', 'Complete',
        'MNAR (Steerage lack of cabin)', 'MCAR (Random omissions)', 'Complete', 'Complete'
    ]
})
display(missing_stats[missing_stats['Missing Count'] > 0])


---
## Question 9: What kind of question can be asked, should we ask. Does the dataset suits the question we are asking?

### Answer:
#### 1. What kind of questions can and should we ask?
With this dataset, we can ask three fundamental categories of analytical and predictive questions:

* **Primary Predictive Question (Supervised Learning):**
  > *"Given a passenger's socioeconomic profile (ticket class, fare), demographic attributes (age, sex), and traveling company (family presence), can we accurately predict their probability of survival during a maritime evacuation?"*

* **Descriptive & Sociological Hypothesis Question:**
  > *"Did the maritime norm of 'Women and Children First' hold equally across all socioeconomic classes, or did upper-class privilege override standard evacuation protocols?"*

* **Feature Interaction & Behavioral Question:**
  > *"Did traveling alone vs. traveling with a large family increase or decrease individual survival rates?"*

---
#### 2. Does the dataset suit the questions we are asking?
**YES, the dataset is exceptionally well-suited to answer these questions for the following reasons:**

1. **Alignment of Observational Unit:**  
   The questions pertain to *individual survival*. Because each row represents one distinct passenger with their individual outcome and personal characteristics, the unit of analysis matches the analytical questions without aggregation bias.
2. **Presence of Key Confounders and Covariates:**  
   The dataset captures both social status (`pclass`, `fare`) and demographic vulnerability (`sex`, `age`, `who`), enabling multivariate regression or machine learning models to isolate interaction effects (e.g., survival rate of 3rd class females vs 1st class males).
3. **Verified Ground Truth:**  
   Unlike surveyed intention data, the outcome (`survived`) is an objective, historically verified factual record.


In [ ]:
# 9.1 Empirical Verification of Analytical Questions
print("=== 1. TESTING 'WOMEN AND CHILDREN FIRST' RULE ===")
display(df.groupby('who')['survived'].agg(['count', 'mean']).rename(columns={'count': 'Passenger Count', 'mean': 'Survival Rate'}))

print("\n=== 2. TESTING CLASS x GENDER INTERACTION ===")
pivot_interaction = df.pivot_table(index='class', columns='sex', values='survived', aggfunc=['count', 'mean'])
display(pivot_interaction)

print("\n=== 3. TESTING FAMILY SIZE EFFECT ON SURVIVAL ===")
df['family_size'] = df['sibsp'] + df['parch'] + 1
display(df.groupby('family_size')['survived'].agg(['count', 'mean']).rename(columns={'count': 'Passenger Count', 'mean': 'Survival Rate'}))


---
## Summary of Tutorial 03 Findings
- **Dataset:** RMS Titanic Passenger Record (891 rows, 15 columns).
- **Quality:** High tidy structure, zero duplicate rows; requires imputation for `age` and categorical handling for `deck`.
- **Sample & Feature Sufficiency:** 891 samples and 9 core independent features are well-sized and well-balanced for binary classification and hypothesis testing.
- **Null Diagnostics:** Missing values exist in exactly 4 columns: `deck` (688), `age` (177), `embarked` (2: Rows 61 & 829), `embark_town` (2: Rows 61 & 829).
- **Target vs Features:** Target is `survived`; features encompass demographics, class, fare, and family structure.
- **Analytical Fit:** Outstanding suitability for predictive modeling and socioeconomic survival disparity research.
